# Chapter 8.2: Implementing a New Model in vLLM -- Step by Step

## Learning Objectives

By the end of this notebook, you will:

1. **Define** a complete model class following vLLM conventions
2. **Implement** the `forward()` method with vLLM's attention backends
3. **Handle** weight loading from HuggingFace checkpoints
4. **Integrate** with PagedAttention and KV cache
5. **Support** tensor parallelism (TP) for multi-GPU inference
6. **Register** and test the model end-to-end
7. **Debug** common implementation pitfalls

---

## Prerequisites
- Chapter 8.1 (Model Registration System)
- Understanding of transformer architecture
- Familiarity with PyTorch `nn.Module`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.2_vllm_new_model.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.2_vllm_new_model.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_model_implementation_checklist():
    """Model implementation checklist flowchart for adding a new model to vLLM."""
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.set_xlim(0, 18)
    ax.set_ylim(0, 4)
    ax.axis('off')
    ax.set_title('New Model Implementation Checklist Flowchart', fontsize=16, fontweight='bold', pad=15)

    steps = [
        ('1. Define\nModel Class', '#4A90D9'),
        ('2. Implement\nforward()', '#27AE60'),
        ('3. Weight\nLoading', '#F39C12'),
        ('4. Attention\nIntegration', '#E74C3C'),
        ('5. Tensor\nParallelism', '#8E44AD'),
        ('6. Register\nModel', '#4A90D9'),
        ('7. Test\nEnd-to-End', '#27AE60'),
    ]

    box_w, box_h = 2.0, 1.8
    y_center = 2.0
    x_start = 0.5
    spacing = 2.4

    for i, (label, color) in enumerate(steps):
        x = x_start + i * spacing
        rect = FancyBboxPatch((x, y_center - box_h / 2), box_w, box_h,
                              boxstyle='round,pad=0.15', facecolor=color,
                              edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x + box_w / 2, y_center, label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')

        # Draw arrow to next step
        if i < len(steps) - 1:
            ax.annotate('', xy=(x + box_w + 0.1, y_center),
                        xytext=(x + box_w + spacing - box_w - 0.1, y_center),
                        arrowprops=dict(arrowstyle='<-', color='#333333', lw=2))

    # Add detail annotations below each box
    details = [
        'nn.Module\nSupportsXX',
        'prefill &\ndecode paths',
        'HF checkpoint\nmapping',
        'PagedAttention\nbackend',
        'Column/Row\nparallel linear',
        'ModelRegistry\n.register()',
        'Compare with\nHF outputs',
    ]
    for i, detail in enumerate(details):
        x = x_start + i * spacing + box_w / 2
        ax.text(x, y_center - box_h / 2 - 0.35, detail, ha='center', va='top',
                fontsize=7, color='#555555', style='italic')

    plt.tight_layout()
    plt.savefig('model_implementation_checklist.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_model_implementation_checklist()

## 1. Architecture Overview: What We're Building

We'll implement a model similar to Qwen2/Gemma in vLLM. The architecture follows the standard decoder-only transformer pattern with these components:

```
+------------------------------------------+
|         MyModelForCausalLM               |
+------------------------------------------+
|                                          |
|  embed_tokens (VocabParallelEmbedding)   |
|       |                                  |
|       v                                  |
|  +----------------------------------+    |
|  | MyDecoderLayer (x num_layers)    |    |
|  |                                  |    |
|  |  input_layernorm (RMSNorm)       |    |
|  |       |                          |    |
|  |  MyAttention                     |    |
|  |    - qkv_proj (QKVParallel)      |    |
|  |    - o_proj (RowParallel)        |    |
|  |    - rotary_emb                  |    |
|  |    - attn (PagedAttention)       |    |
|  |       |                          |    |
|  |  post_attn_layernorm (RMSNorm)   |    |
|  |       |                          |    |
|  |  MyMLP                           |    |
|  |    - gate_up_proj (MergedCol)    |    |
|  |    - down_proj (RowParallel)     |    |
|  |    - act_fn (SiLU)              |    |
|  +----------------------------------+    |
|       |                                  |
|  norm (RMSNorm)                          |
|       |                                  |
|  lm_head (ParallelLMHead)               |
|       |                                  |
|  logits_processor                        |
+------------------------------------------+
```

## 2. Step 1: Understanding vLLM's Layer Primitives

Before implementing the model, we need to understand the special layer types vLLM provides. These replace standard PyTorch layers with versions optimized for:
- Tensor Parallelism
- Quantization
- Fused kernels

In [ ]:
# vLLM provides specialized linear layers for tensor parallelism
# These are in vllm/model_executor/layers/

# Let's understand each one:

print("vLLM Layer Primitives")
print("=" * 60)

layers = {
    "ColumnParallelLinear": {
        "description": "Splits OUTPUT dimension across GPUs",
        "usage": "For Q, K, V projections (each GPU gets a subset of heads)",
        "example": "Linear(4096, 4096) with TP=2 -> each GPU has Linear(4096, 2048)",
        "import": "from vllm.model_executor.layers.linear import ColumnParallelLinear",
    },
    "RowParallelLinear": {
        "description": "Splits INPUT dimension across GPUs",
        "usage": "For O projection and down_proj (gathers results across GPUs)",
        "example": "Linear(4096, 4096) with TP=2 -> each GPU has Linear(2048, 4096)",
        "import": "from vllm.model_executor.layers.linear import RowParallelLinear",
    },
    "QKVParallelLinear": {
        "description": "Column-parallel for Q, K, V with proper head splitting",
        "usage": "Merged Q+K+V projection with correct GQA handling",
        "example": "Handles different num_heads for Q vs K/V (GQA)",
        "import": "from vllm.model_executor.layers.linear import QKVParallelLinear",
    },
    "MergedColumnParallelLinear": {
        "description": "Column-parallel for merged gate+up projections",
        "usage": "For SwiGLU FFN where gate and up are merged",
        "example": "gate_proj + up_proj merged into one linear layer",
        "import": "from vllm.model_executor.layers.linear import MergedColumnParallelLinear",
    },
    "VocabParallelEmbedding": {
        "description": "Embedding table split across GPUs by vocabulary",
        "usage": "Token embedding layer",
        "example": "Embedding(32000, 4096) with TP=2 -> each GPU has half vocab",
        "import": "from vllm.model_executor.layers.vocab_parallel_embedding import VocabParallelEmbedding",
    },
    "ParallelLMHead": {
        "description": "Output projection (logits) with TP",
        "usage": "Final linear layer to vocab size",
        "example": "Linear(4096, 32000) split across GPUs",
        "import": "from vllm.model_executor.layers.vocab_parallel_embedding import ParallelLMHead",
    },
    "RMSNorm": {
        "description": "Root Mean Square Layer Normalization",
        "usage": "Used in Llama/Qwen/Gemma instead of LayerNorm",
        "example": "Fused CUDA kernel for efficiency",
        "import": "from vllm.model_executor.layers.layernorm import RMSNorm",
    },
    "RotaryEmbedding": {
        "description": "Rotary Position Embedding (RoPE)",
        "usage": "Position encoding for attention",
        "example": "Supports various RoPE variants (linear, dynamic, YaRN)",
        "import": "from vllm.model_executor.layers.rotary_embedding import get_rope",
    },
}

for name, info in layers.items():
    print(f"\n{name}:")
    print(f"  Description: {info['description']}")
    print(f"  Usage: {info['usage']}")
    print(f"  Example: {info['example']}")
    print(f"  Import: {info['import']}")

## 3. Step 2: Define the MLP (Feed-Forward Network)

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, List, Iterable, Tuple, Set

# In real vLLM, these would be imported from vllm.model_executor.layers
# For this tutorial, we simulate them

class SimulatedMergedColumnParallelLinear(nn.Module):
    """Simulates MergedColumnParallelLinear for tutorial purposes."""
    def __init__(self, input_size, output_sizes, bias=False, quant_config=None):
        super().__init__()
        total_output = sum(output_sizes)
        self.linear = nn.Linear(input_size, total_output, bias=bias)
        self.output_sizes = output_sizes
    
    def forward(self, x):
        return self.linear(x), None  # (output, all_reduce_needed)

class SimulatedRowParallelLinear(nn.Module):
    """Simulates RowParallelLinear for tutorial purposes."""
    def __init__(self, input_size, output_size, bias=False, quant_config=None):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size, bias=bias)
    
    def forward(self, x):
        return self.linear(x), None


class MyMLP(nn.Module):
    """
    SwiGLU MLP (used in Llama/Qwen/Gemma).
    
    Architecture:
        gate = sigmoid(gate_proj(x))
        up = up_proj(x) 
        output = down_proj(gate * up)
    
    In vLLM, gate_proj and up_proj are MERGED into gate_up_proj
    for a single matrix multiply.
    """
    
    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        hidden_act: str = "silu",
        quant_config=None,
    ):
        super().__init__()
        
        # In real vLLM:
        # self.gate_up_proj = MergedColumnParallelLinear(
        #     hidden_size, [intermediate_size] * 2, bias=False,
        #     quant_config=quant_config
        # )
        self.gate_up_proj = SimulatedMergedColumnParallelLinear(
            hidden_size, [intermediate_size, intermediate_size],
            bias=False, quant_config=quant_config
        )
        
        # In real vLLM:
        # self.down_proj = RowParallelLinear(
        #     intermediate_size, hidden_size, bias=False,
        #     quant_config=quant_config
        # )
        self.down_proj = SimulatedRowParallelLinear(
            intermediate_size, hidden_size,
            bias=False, quant_config=quant_config
        )
        
        # Activation function
        if hidden_act == "silu":
            self.act_fn = nn.SiLU()
        elif hidden_act == "gelu":
            self.act_fn = nn.GELU()
        else:
            raise ValueError(f"Unsupported activation: {hidden_act}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Merged gate + up projection
        gate_up, _ = self.gate_up_proj(x)
        
        # Split into gate and up
        intermediate_size = gate_up.shape[-1] // 2
        gate = gate_up[..., :intermediate_size]
        up = gate_up[..., intermediate_size:]
        
        # SwiGLU: silu(gate) * up
        x = self.act_fn(gate) * up
        
        # Down projection
        x, _ = self.down_proj(x)
        return x

# Test MLP
mlp = MyMLP(hidden_size=256, intermediate_size=512)
x = torch.randn(2, 8, 256)  # [batch, seq, hidden]
out = mlp(x)
print(f"MLP input:  {x.shape}")
print(f"MLP output: {out.shape}")
print(f"MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")

## 4. Step 3: Define the Attention Layer

The attention layer is the most complex part because it must integrate with vLLM's PagedAttention and KV cache system.

In [ ]:
class SimulatedQKVParallelLinear(nn.Module):
    """Simulates QKVParallelLinear."""
    def __init__(self, hidden_size, head_size, num_heads, num_kv_heads,
                 bias=False, quant_config=None):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_size = head_size
        q_size = num_heads * head_size
        kv_size = 2 * num_kv_heads * head_size
        self.linear = nn.Linear(hidden_size, q_size + kv_size, bias=bias)
    
    def forward(self, x):
        return self.linear(x), None


class MyAttention(nn.Module):
    """
    Multi-Head Attention with Grouped Query Attention (GQA) support.
    
    Key features:
    1. QKV are merged into a single projection for efficiency
    2. Supports GQA (num_kv_heads < num_heads)
    3. Integrates with vLLM's PagedAttention via Attention wrapper
    4. Uses RoPE for position encoding
    
    Architecture:
        qkv = qkv_proj(hidden_states)     # Single matmul
        q, k, v = split(qkv)              # Split Q, K, V
        q, k = rotary_emb(q, k, positions) # Apply RoPE
        output = paged_attention(q, k, v)   # PagedAttention
        output = o_proj(output)             # Output projection
    """
    
    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        num_kv_heads: int,
        max_position_embeddings: int = 4096,
        rope_theta: float = 10000.0,
        quant_config=None,
        cache_config=None,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_heads
        
        # QKV projection (merged)
        # In real vLLM:
        # self.qkv_proj = QKVParallelLinear(
        #     hidden_size, self.head_dim, num_heads, num_kv_heads,
        #     bias=False, quant_config=quant_config
        # )
        self.qkv_proj = SimulatedQKVParallelLinear(
            hidden_size, self.head_dim, num_heads, num_kv_heads,
            bias=False, quant_config=quant_config
        )
        
        # Output projection
        self.o_proj = SimulatedRowParallelLinear(
            hidden_size, hidden_size, bias=False, quant_config=quant_config
        )
        
        # Rotary embedding
        # In real vLLM:
        # self.rotary_emb = get_rope(
        #     self.head_dim, rotary_dim=self.head_dim,
        #     max_position=max_position_embeddings,
        #     base=rope_theta,
        # )
        
        # Attention backend
        # In real vLLM:
        # self.attn = Attention(
        #     num_heads, self.head_dim, scale=self.head_dim**-0.5,
        #     num_kv_heads=num_kv_heads, cache_config=cache_config,
        # )
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        positions: torch.Tensor,
        kv_cache: Optional[torch.Tensor] = None,
        attn_metadata=None,
    ) -> torch.Tensor:
        """
        Forward pass for attention.
        
        In vLLM, the key difference from standard PyTorch attention is:
        1. input_ids is 1D (all sequences concatenated)
        2. KV cache is managed by PagedAttention
        3. The Attention wrapper handles prefill vs decode
        """
        # Step 1: QKV projection
        qkv, _ = self.qkv_proj(hidden_states)
        
        # Step 2: Split Q, K, V
        q_size = self.num_heads * self.head_dim
        kv_size = self.num_kv_heads * self.head_dim
        q = qkv[..., :q_size]
        k = qkv[..., q_size:q_size + kv_size]
        v = qkv[..., q_size + kv_size:]
        
        # Step 3: Apply RoPE (simplified - in real vLLM, this is fused)
        # q, k = self.rotary_emb(positions, q, k)
        
        # Step 4: PagedAttention
        # In real vLLM:
        # attn_output = self.attn(q, k, v, kv_cache, attn_metadata)
        
        # Simplified: standard attention for tutorial
        batch_size = hidden_states.shape[0] if hidden_states.dim() > 1 else 1
        seq_len = hidden_states.shape[-2] if hidden_states.dim() > 1 else hidden_states.shape[0]
        
        q = q.view(-1, self.num_heads, self.head_dim)
        k = k.view(-1, self.num_kv_heads, self.head_dim)
        v = v.view(-1, self.num_kv_heads, self.head_dim)
        
        # Simple dot-product attention (not PagedAttention)
        attn_output = q  # Placeholder for actual attention
        attn_output = attn_output.reshape(-1, self.hidden_size)
        
        # Step 5: Output projection
        output, _ = self.o_proj(attn_output)
        return output

# Test attention
attn = MyAttention(
    hidden_size=256, num_heads=4, num_kv_heads=2  # GQA: 4 Q heads, 2 KV heads
)
x = torch.randn(8, 256)  # [num_tokens, hidden_size]
positions = torch.arange(8)
out = attn(x, positions)
print(f"Attention input:  {x.shape}")
print(f"Attention output: {out.shape}")
print(f"GQA ratio: {attn.num_heads // attn.num_kv_heads}x (4 Q heads share 2 KV heads)")

## 5. Step 4: Define the Decoder Layer

In [ ]:
class MyDecoderLayer(nn.Module):
    """
    A single transformer decoder layer.
    
    Structure:
        residual = x
        x = input_layernorm(x)
        x = self_attn(x)
        x = residual + x
        
        residual = x
        x = post_attention_layernorm(x)
        x = mlp(x)
        x = residual + x
    """
    
    def __init__(
        self,
        config,
        layer_idx: int,
        cache_config=None,
        quant_config=None,
    ):
        super().__init__()
        self.hidden_size = config.hidden_size
        
        # Self attention
        self.self_attn = MyAttention(
            hidden_size=config.hidden_size,
            num_heads=config.num_attention_heads,
            num_kv_heads=getattr(config, 'num_key_value_heads', config.num_attention_heads),
            max_position_embeddings=getattr(config, 'max_position_embeddings', 4096),
            rope_theta=getattr(config, 'rope_theta', 10000.0),
            quant_config=quant_config,
            cache_config=cache_config,
        )
        
        # MLP
        self.mlp = MyMLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=getattr(config, 'hidden_act', 'silu'),
            quant_config=quant_config,
        )
        
        # Layer norms
        # In real vLLM: RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.input_layernorm = nn.LayerNorm(config.hidden_size)
        self.post_attention_layernorm = nn.LayerNorm(config.hidden_size)
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        positions: torch.Tensor,
        kv_cache: Optional[torch.Tensor] = None,
        attn_metadata=None,
        residual: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for decoder layer.
        
        Note: In vLLM, some models use fused add+norm for efficiency.
        The residual connection might be passed between layers.
        """
        # Pre-attention norm
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states = self.input_layernorm(hidden_states)
            # In real vLLM with fused kernels:
            # hidden_states, residual = self.input_layernorm(hidden_states, residual)
        
        # Self attention
        hidden_states = self.self_attn(
            hidden_states, positions, kv_cache, attn_metadata
        )
        
        # Residual connection
        hidden_states = residual + hidden_states
        
        # Post-attention norm
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        
        # MLP
        hidden_states = self.mlp(hidden_states)
        
        # Residual connection
        hidden_states = residual + hidden_states
        
        return hidden_states, residual

# Test decoder layer
class MockConfig:
    hidden_size = 256
    num_attention_heads = 4
    num_key_value_heads = 2
    intermediate_size = 512
    max_position_embeddings = 4096
    rope_theta = 10000.0
    hidden_act = 'silu'

config = MockConfig()
layer = MyDecoderLayer(config, layer_idx=0)
x = torch.randn(8, 256)
positions = torch.arange(8)
out, residual = layer(x, positions)
print(f"Decoder layer input:    {x.shape}")
print(f"Decoder layer output:   {out.shape}")
print(f"Decoder layer residual: {residual.shape}")

## 6. Step 5: Define the Full Model

In [ ]:
class MyModel(nn.Module):
    """
    The transformer backbone (without lm_head).
    
    This follows the pattern of LlamaModel in vLLM:
    embed_tokens -> decoder_layers -> final_norm
    """
    
    def __init__(self, config, cache_config=None, quant_config=None):
        super().__init__()
        self.config = config
        self.padding_idx = getattr(config, 'pad_token_id', None)
        
        # Token embeddings
        # In real vLLM: VocabParallelEmbedding(config.vocab_size, config.hidden_size)
        self.embed_tokens = nn.Embedding(
            config.vocab_size, config.hidden_size, padding_idx=self.padding_idx
        )
        
        # Decoder layers
        self.layers = nn.ModuleList([
            MyDecoderLayer(config, layer_idx=i,
                          cache_config=cache_config,
                          quant_config=quant_config)
            for i in range(config.num_hidden_layers)
        ])
        
        # Final layer norm
        # In real vLLM: RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.norm = nn.LayerNorm(config.hidden_size)
    
    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
        kv_caches: List[torch.Tensor],
        attn_metadata=None,
        intermediate_tensors=None,  # For pipeline parallelism
        inputs_embeds: Optional[torch.Tensor] = None,  # For multimodal
    ) -> torch.Tensor:
        """
        Forward pass through the transformer backbone.
        
        Returns: hidden_states [num_tokens, hidden_size]
        """
        # Get embeddings
        if inputs_embeds is not None:
            hidden_states = inputs_embeds
        else:
            hidden_states = self.embed_tokens(input_ids)
        
        # Process through layers
        residual = None
        for i, layer in enumerate(self.layers):
            kv_cache = kv_caches[i] if kv_caches else None
            hidden_states, residual = layer(
                hidden_states, positions, kv_cache, attn_metadata, residual
            )
        
        # Final norm
        hidden_states = self.norm(hidden_states)
        
        return hidden_states

print("MyModel (transformer backbone) defined successfully.")

In [ ]:
class MyModelForCausalLM(nn.Module):
    """
    Complete causal language model for vLLM.
    
    This is the top-level class that gets registered with ModelRegistry.
    It wraps MyModel and adds:
    - LM head (output projection to vocabulary)
    - Logits processing
    - Weight loading from HuggingFace
    
    Equivalent to LlamaForCausalLM in vLLM.
    """
    
    # LoRA support (optional)
    packed_modules_mapping = {
        "qkv_proj": ["q_proj", "k_proj", "v_proj"],
        "gate_up_proj": ["gate_proj", "up_proj"],
    }
    supported_lora_modules = [
        "qkv_proj", "o_proj", "gate_up_proj", "down_proj",
    ]
    embedding_modules = {}
    embedding_padding_modules = []
    
    def __init__(
        self,
        config,
        cache_config=None,
        quant_config=None,
        lora_config=None,
    ):
        super().__init__()
        self.config = config
        self.quant_config = quant_config
        
        # The transformer model
        self.model = MyModel(config, cache_config, quant_config)
        
        # LM head (output projection)
        # In real vLLM: ParallelLMHead(config.vocab_size, config.hidden_size)
        self.lm_head = nn.Linear(
            config.hidden_size, config.vocab_size, bias=False
        )
        
        # In real vLLM:
        # self.logits_processor = LogitsProcessor(config.vocab_size)
        # self.sampler = Sampler()
    
    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
        kv_caches: List[torch.Tensor],
        attn_metadata=None,
        intermediate_tensors=None,
        inputs_embeds: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass.
        
        Returns logits [num_tokens, vocab_size]
        """
        # Get hidden states from transformer
        hidden_states = self.model(
            input_ids, positions, kv_caches, attn_metadata,
            intermediate_tensors, inputs_embeds
        )
        
        # Compute logits
        # In real vLLM:
        # logits = self.logits_processor(self.lm_head, hidden_states, sampling_metadata)
        logits = self.lm_head(hidden_states)
        
        return logits
    
    def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]) -> Set[str]:
        """
        Load weights from HuggingFace checkpoint.
        
        This method handles:
        1. Weight name remapping (HF -> vLLM)
        2. Merging separate Q/K/V into QKV
        3. Merging gate/up into gate_up
        4. Tensor parallelism sharding
        """
        stacked_params_mapping = [
            # (vllm_param_name, hf_shard_name, shard_id)
            ("qkv_proj", "q_proj", "q"),
            ("qkv_proj", "k_proj", "k"),
            ("qkv_proj", "v_proj", "v"),
            ("gate_up_proj", "gate_proj", 0),
            ("gate_up_proj", "up_proj", 1),
        ]
        
        params_dict = dict(self.named_parameters())
        loaded_params: Set[str] = set()
        
        for name, loaded_weight in weights:
            # Check if this is a stacked parameter
            for (vllm_name, shard_name, shard_id) in stacked_params_mapping:
                if shard_name not in name:
                    continue
                
                # Replace HF shard name with vLLM merged name
                param_name = name.replace(shard_name, vllm_name)
                
                if param_name not in params_dict:
                    break
                
                param = params_dict[param_name]
                # In real vLLM: weight_loader = param.weight_loader
                # weight_loader(param, loaded_weight, shard_id)
                
                loaded_params.add(name)
                break
            else:
                # Direct parameter (no merging needed)
                if name in params_dict:
                    param = params_dict[name]
                    # In real vLLM, handles quantized weights too
                    param.data.copy_(loaded_weight)
                    loaded_params.add(name)
        
        return loaded_params

# Test the full model
class FullConfig:
    vocab_size = 1000
    hidden_size = 256
    num_attention_heads = 4
    num_key_value_heads = 2
    num_hidden_layers = 4
    intermediate_size = 512
    max_position_embeddings = 4096
    rope_theta = 10000.0
    hidden_act = 'silu'

config = FullConfig()
model = MyModelForCausalLM(config)

# Forward pass
input_ids = torch.randint(0, config.vocab_size, (16,))  # 16 tokens
positions = torch.arange(16)
logits = model(input_ids, positions, kv_caches=[])

print(f"Model: MyModelForCausalLM")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input:  {input_ids.shape} (token IDs)")
print(f"Output: {logits.shape} (logits)")

## 7. Step 6: Integrate with vLLM's Attention Backend

In real vLLM, you don't implement attention manually. Instead, you use the `Attention` wrapper which dispatches to the appropriate backend (FlashAttention, FlashInfer, etc.).

In [ ]:
# The vLLM Attention wrapper
# File: vllm/attention/layer.py

ATTENTION_INTEGRATION_CODE = '''
# In your attention class, use the Attention wrapper:

from vllm.attention import Attention

class MyAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, num_kv_heads, ...):
        super().__init__()
        self.head_dim = hidden_size // num_heads
        
        # QKV and output projections (as before)
        self.qkv_proj = QKVParallelLinear(...)
        self.o_proj = RowParallelLinear(...)
        
        # RoPE
        self.rotary_emb = get_rope(
            self.head_dim,
            rotary_dim=self.head_dim,
            max_position=max_position_embeddings,
            base=rope_theta,
        )
        
        # THE KEY PART: Use vLLM's Attention wrapper
        self.attn = Attention(
            num_heads=num_heads,
            head_size=self.head_dim,
            scale=self.head_dim ** -0.5,
            num_kv_heads=num_kv_heads,
            cache_config=cache_config,
            quant_config=quant_config,
        )
    
    def forward(self, hidden_states, positions, kv_cache, attn_metadata):
        # QKV projection
        qkv, _ = self.qkv_proj(hidden_states)
        q, k, v = qkv.split([q_size, kv_size, kv_size], dim=-1)
        
        # Apply RoPE
        q, k = self.rotary_emb(positions, q, k)
        
        # Attention (PagedAttention with KV cache)
        # This ONE line handles:
        # - KV cache read/write
        # - Prefill vs decode dispatch
        # - Backend selection (FlashAttention/FlashInfer/etc.)
        # - Paged memory management
        attn_output = self.attn(q, k, v, kv_cache, attn_metadata)
        
        # Output projection
        output, _ = self.o_proj(attn_output)
        return output
'''

print(ATTENTION_INTEGRATION_CODE)
print("\nKey insight: The Attention wrapper abstracts away ALL of the")
print("complexity of PagedAttention, KV cache, and attention backends.")
print("You just call self.attn(q, k, v, kv_cache, attn_metadata)!")

## 8. Step 7: Tensor Parallelism Support

Tensor Parallelism (TP) splits model weights across multiple GPUs. vLLM's parallel linear layers handle this automatically.

In [ ]:
# How Tensor Parallelism works in vLLM

print("Tensor Parallelism in vLLM")
print("=" * 60)

print("""
ColumnParallelLinear (splits output dimension):
    Full: [hidden, hidden] -> split into [hidden, hidden/TP] per GPU
    
    GPU 0: weight[0:hidden/2, :] * input -> partial_output_0
    GPU 1: weight[hidden/2:, :] * input -> partial_output_1
    
    Each GPU has complete input, partial output

RowParallelLinear (splits input dimension):
    Full: [hidden, hidden] -> split into [hidden/TP, hidden] per GPU
    
    GPU 0: weight[:, 0:hidden/2] * partial_input_0 -> full_output_0
    GPU 1: weight[:, hidden/2:] * partial_input_1 -> full_output_1
    
    full_output = all_reduce(full_output_0 + full_output_1)
    Each GPU has partial input, needs all-reduce for complete output

Pattern: Column -> Row (no communication between them)
    Attention: QKV(Column) -> O(Row)    [all-reduce at O output]
    MLP:       gate_up(Column) -> down(Row)  [all-reduce at down output]
""")

# Visual example
print("\nExample with TP=2, hidden_size=4096:")
print("-" * 40)

tp_size = 2
hidden_size = 4096
num_heads = 32
num_kv_heads = 8
intermediate_size = 11008

print(f"\nQKV Projection (ColumnParallel):")
q_per_gpu = num_heads // tp_size
kv_per_gpu = num_kv_heads // tp_size
head_dim = hidden_size // num_heads
print(f"  Full: [{hidden_size}, {(num_heads + 2*num_kv_heads) * head_dim}]")
print(f"  Per GPU: [{hidden_size}, {(q_per_gpu + 2*kv_per_gpu) * head_dim}]")
print(f"  Q heads per GPU: {q_per_gpu}, KV heads per GPU: {kv_per_gpu}")

print(f"\nO Projection (RowParallel):")
print(f"  Full: [{hidden_size}, {hidden_size}]")
print(f"  Per GPU: [{hidden_size // tp_size}, {hidden_size}]")

print(f"\nGate+Up Projection (MergedColumn):")
print(f"  Full: [{hidden_size}, {2 * intermediate_size}]")
print(f"  Per GPU: [{hidden_size}, {2 * intermediate_size // tp_size}]")

print(f"\nDown Projection (RowParallel):")
print(f"  Full: [{intermediate_size}, {hidden_size}]")
print(f"  Per GPU: [{intermediate_size // tp_size}, {hidden_size}]")

## 9. Step 8: Weight Loading from HuggingFace

Let's implement a detailed weight loading method that handles all the common patterns.

In [ ]:
def detailed_load_weights(
    model: nn.Module,
    weights: Iterable[Tuple[str, torch.Tensor]],
) -> Set[str]:
    """
    Detailed weight loading implementation.
    
    This shows exactly what happens when vLLM loads HuggingFace weights.
    """
    
    # Step 1: Define stacked parameter mappings
    stacked_params_mapping = [
        # (vllm_merged_name, hf_shard_name, shard_id)
        ("qkv_proj", "q_proj", "q"),
        ("qkv_proj", "k_proj", "k"),
        ("qkv_proj", "v_proj", "v"),
        ("gate_up_proj", "gate_proj", 0),
        ("gate_up_proj", "up_proj", 1),
    ]
    
    # Step 2: Get model parameters
    params_dict = dict(model.named_parameters())
    loaded_params: Set[str] = set()
    
    for name, loaded_weight in weights:
        print(f"\nProcessing: {name} (shape: {list(loaded_weight.shape)})")
        
        # Step 3: Handle special weight name patterns
        
        # Some models use 'model.layers' while others use 'transformer.h'
        # Add remapping here if needed:
        # name = name.replace("transformer.h.", "model.layers.")
        
        # Step 4: Check if this is a stacked (merged) parameter
        is_stacked = False
        for (vllm_name, shard_name, shard_id) in stacked_params_mapping:
            if shard_name not in name:
                continue
            
            # Construct the vLLM parameter name
            param_name = name.replace(shard_name, vllm_name)
            
            if param_name not in params_dict:
                print(f"  SKIP: merged name '{param_name}' not in model")
                break
            
            param = params_dict[param_name]
            print(f"  MERGE: {name} -> {param_name} (shard={shard_id})")
            
            # In real vLLM, the weight_loader handles tensor parallelism:
            # param.weight_loader(param, loaded_weight, shard_id)
            
            loaded_params.add(name)
            is_stacked = True
            break
        
        if is_stacked:
            continue
        
        # Step 5: Direct parameter copy
        if name in params_dict:
            param = params_dict[name]
            if param.shape != loaded_weight.shape:
                print(f"  ERROR: Shape mismatch! Model: {list(param.shape)}, Weight: {list(loaded_weight.shape)}")
                continue
            param.data.copy_(loaded_weight)
            print(f"  COPY: {name}")
            loaded_params.add(name)
        else:
            print(f"  SKIP: '{name}' not in model parameters")
    
    # Step 6: Report missing weights
    all_params = set(params_dict.keys())
    # For merged params, we need to check differently
    print(f"\n{'='*60}")
    print(f"Loaded {len(loaded_params)} weights")
    print(f"Model has {len(all_params)} parameters")
    
    return loaded_params

# Simulate weight loading
print("Simulating weight loading from HuggingFace checkpoint")
print("=" * 60)

fake_hf_weights = [
    ("model.embed_tokens.weight", torch.randn(1000, 256)),
    ("model.layers.0.self_attn.q_proj.weight", torch.randn(256, 256)),
    ("model.layers.0.self_attn.k_proj.weight", torch.randn(128, 256)),
    ("model.layers.0.self_attn.v_proj.weight", torch.randn(128, 256)),
    ("model.layers.0.self_attn.o_proj.weight", torch.randn(256, 256)),
    ("model.layers.0.mlp.gate_proj.weight", torch.randn(512, 256)),
    ("model.layers.0.mlp.up_proj.weight", torch.randn(512, 256)),
    ("model.layers.0.mlp.down_proj.weight", torch.randn(256, 512)),
]

loaded = detailed_load_weights(model, fake_hf_weights)

## 10. Step 9: Register and Test

In [ ]:
# Complete registration and testing workflow

REGISTRATION_CODE = '''
# File: my_model.py
# Contains the complete model implementation (shown above)

# Step 1: Register the model
from vllm import ModelRegistry
from my_model import MyModelForCausalLM

ModelRegistry.register_model("MyModelForCausalLM", MyModelForCausalLM)

# Step 2: Create a config.json for your model
config_json = {
    "architectures": ["MyModelForCausalLM"],
    "hidden_size": 4096,
    "intermediate_size": 11008,
    "num_attention_heads": 32,
    "num_key_value_heads": 8,
    "num_hidden_layers": 32,
    "vocab_size": 32000,
    "max_position_embeddings": 4096,
    "rms_norm_eps": 1e-6,
    "rope_theta": 10000.0,
    "hidden_act": "silu",
}

# Step 3: Test with vLLM
from vllm import LLM, SamplingParams

llm = LLM(model="/path/to/my_model")
outputs = llm.generate(["Hello, world!"], SamplingParams(max_tokens=50))
print(outputs[0].outputs[0].text)
'''

print(REGISTRATION_CODE)

## 11. Full Worked Example: A Qwen2-style Model

Let's put everything together with a complete Qwen2-style model implementation.

In [ ]:
# Complete Qwen2-style model for vLLM
# This shows the REAL structure with all necessary components

COMPLETE_MODEL_CODE = '''
"""Qwen2-style model for vLLM."""

import torch
import torch.nn as nn
from typing import Iterable, List, Optional, Set, Tuple

from vllm.attention import Attention
from vllm.config import CacheConfig
from vllm.model_executor.layers.activation import SiluAndMul
from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.layers.linear import (
    MergedColumnParallelLinear,
    QKVParallelLinear,
    RowParallelLinear,
)
from vllm.model_executor.layers.logits_processor import LogitsProcessor
from vllm.model_executor.layers.quantization import QuantizationConfig
from vllm.model_executor.layers.rotary_embedding import get_rope
from vllm.model_executor.layers.sampler import Sampler, SamplerOutput
from vllm.model_executor.layers.vocab_parallel_embedding import (
    ParallelLMHead,
    VocabParallelEmbedding,
)
from vllm.model_executor.model_loader.weight_utils import default_weight_loader
from vllm.model_executor.sampling_metadata import SamplingMetadata
from vllm.sequence import IntermediateTensors


class Qwen2MLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size, hidden_act, quant_config):
        super().__init__()
        self.gate_up_proj = MergedColumnParallelLinear(
            hidden_size, [intermediate_size] * 2,
            bias=False, quant_config=quant_config,
        )
        self.down_proj = RowParallelLinear(
            intermediate_size, hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.act_fn = SiluAndMul()  # Fused SiLU + element-wise multiply

    def forward(self, x):
        gate_up, _ = self.gate_up_proj(x)
        x = self.act_fn(gate_up)
        x, _ = self.down_proj(x)
        return x


class Qwen2Attention(nn.Module):
    def __init__(self, hidden_size, num_heads, num_kv_heads,
                 max_position, rope_theta, cache_config, quant_config,
                 bias=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.head_dim = hidden_size // num_heads
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.scaling = self.head_dim ** -0.5

        self.qkv_proj = QKVParallelLinear(
            hidden_size, self.head_dim, num_heads, num_kv_heads,
            bias=bias, quant_config=quant_config,
        )
        self.o_proj = RowParallelLinear(
            num_heads * self.head_dim, hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.rotary_emb = get_rope(
            self.head_dim, rotary_dim=self.head_dim,
            max_position=max_position, base=rope_theta,
        )
        self.attn = Attention(
            self.num_heads, self.head_dim, self.scaling,
            num_kv_heads=self.num_kv_heads,
            cache_config=cache_config,
            quant_config=quant_config,
        )

    def forward(self, hidden_states, positions, kv_cache, attn_metadata):
        qkv, _ = self.qkv_proj(hidden_states)
        q, k, v = qkv.split(
            [self.num_heads * self.head_dim,
             self.num_kv_heads * self.head_dim,
             self.num_kv_heads * self.head_dim],
            dim=-1
        )
        q, k = self.rotary_emb(positions, q, k)
        attn_output = self.attn(q, k, v, kv_cache, attn_metadata)
        output, _ = self.o_proj(attn_output)
        return output


class Qwen2DecoderLayer(nn.Module):
    def __init__(self, config, layer_idx, cache_config, quant_config):
        super().__init__()
        self.self_attn = Qwen2Attention(
            hidden_size=config.hidden_size,
            num_heads=config.num_attention_heads,
            num_kv_heads=config.num_key_value_heads,
            max_position=config.max_position_embeddings,
            rope_theta=config.rope_theta,
            cache_config=cache_config,
            quant_config=quant_config,
            bias=True,  # Qwen2 uses bias in attention
        )
        self.mlp = Qwen2MLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
            quant_config=quant_config,
        )
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, positions, kv_cache, attn_metadata, residual):
        # Fused add + RMSNorm
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)
        
        hidden_states = self.self_attn(hidden_states, positions, kv_cache, attn_metadata)
        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class Qwen2Model(nn.Module):
    def __init__(self, config, cache_config, quant_config):
        super().__init__()
        self.embed_tokens = VocabParallelEmbedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            Qwen2DecoderLayer(config, i, cache_config, quant_config)
            for i in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, input_ids, positions, kv_caches, attn_metadata,
                intermediate_tensors=None, inputs_embeds=None):
        hidden_states = inputs_embeds if inputs_embeds is not None else self.embed_tokens(input_ids)
        residual = None
        for i, layer in enumerate(self.layers):
            hidden_states, residual = layer(
                hidden_states, positions, kv_caches[i], attn_metadata, residual
            )
        hidden_states, _ = self.norm(hidden_states, residual)
        return hidden_states


class Qwen2ForCausalLM(nn.Module):
    """Top-level model class registered with vLLM."""
    
    packed_modules_mapping = {
        "qkv_proj": ["q_proj", "k_proj", "v_proj"],
        "gate_up_proj": ["gate_proj", "up_proj"],
    }
    supported_lora_modules = [
        "qkv_proj", "o_proj", "gate_up_proj", "down_proj",
    ]
    embedding_modules = {}
    embedding_padding_modules = []

    def __init__(self, config, cache_config=None, quant_config=None, lora_config=None):
        super().__init__()
        self.config = config
        self.model = Qwen2Model(config, cache_config, quant_config)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size, bias=False)
        self.logits_processor = LogitsProcessor(config.vocab_size)
        self.sampler = Sampler()

    def forward(self, input_ids, positions, kv_caches, attn_metadata,
                intermediate_tensors=None, inputs_embeds=None):
        hidden_states = self.model(input_ids, positions, kv_caches, attn_metadata,
                                   intermediate_tensors, inputs_embeds)
        return hidden_states

    def compute_logits(self, hidden_states, sampling_metadata):
        logits = self.logits_processor(self.lm_head, hidden_states, sampling_metadata)
        return logits

    def sample(self, logits, sampling_metadata):
        next_tokens = self.sampler(logits, sampling_metadata)
        return next_tokens

    def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]) -> Set[str]:
        stacked_params_mapping = [
            ("qkv_proj", "q_proj", "q"),
            ("qkv_proj", "k_proj", "k"),
            ("qkv_proj", "v_proj", "v"),
            ("gate_up_proj", "gate_proj", 0),
            ("gate_up_proj", "up_proj", 1),
        ]
        params_dict = dict(self.named_parameters())
        loaded_params: Set[str] = set()
        
        for name, loaded_weight in weights:
            if "rotary_emb.inv_freq" in name:
                continue  # Skip RoPE frequencies (computed at init)
            
            for (param_name, weight_name, shard_id) in stacked_params_mapping:
                if weight_name not in name:
                    continue
                name = name.replace(weight_name, param_name)
                if name not in params_dict:
                    break
                param = params_dict[name]
                weight_loader = param.weight_loader
                weight_loader(param, loaded_weight, shard_id)
                loaded_params.add(name)
                break
            else:
                if name not in params_dict:
                    continue
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", default_weight_loader)
                weight_loader(param, loaded_weight)
                loaded_params.add(name)
        
        return loaded_params
'''

print(COMPLETE_MODEL_CODE)

## 12. Common Pitfalls and Debugging Tips

In [ ]:
pitfalls = [
    {
        "pitfall": "Forgetting to skip rotary_emb.inv_freq in load_weights",
        "symptom": "Weight loading error or warning about unexpected keys",
        "fix": "Add 'if \"rotary_emb.inv_freq\" in name: continue'",
    },
    {
        "pitfall": "Wrong QKV split sizes for GQA",
        "symptom": "Shape mismatch in attention computation",
        "fix": "Q has num_heads * head_dim, K/V each have num_kv_heads * head_dim",
    },
    {
        "pitfall": "Not using vLLM's Attention wrapper",
        "symptom": "KV cache not being used, decoding is slow",
        "fix": "Always use self.attn = Attention(...) from vllm.attention",
    },
    {
        "pitfall": "Using nn.Linear instead of ColumnParallelLinear/RowParallelLinear",
        "symptom": "Model works on 1 GPU but fails with TP > 1",
        "fix": "Use vLLM's parallel linear layers for all projections",
    },
    {
        "pitfall": "Missing bias parameter in QKV projection",
        "symptom": "Weight loading silently loads wrong values",
        "fix": "Check if the HuggingFace model uses bias (varies by model)",
    },
    {
        "pitfall": "Forward method signature doesn't match",
        "symptom": "TypeError or unexpected behavior",
        "fix": "Use exact signature: forward(input_ids, positions, kv_caches, attn_metadata, ...)",
    },
    {
        "pitfall": "Not handling tie_word_embeddings",
        "symptom": "lm_head weights are zeros or wrong",
        "fix": "Check config.tie_word_embeddings and share embed_tokens weights",
    },
    {
        "pitfall": "Incorrect intermediate_size for SwiGLU",
        "symptom": "Shape mismatch in MLP",
        "fix": "SwiGLU uses 2/3 * 4 * hidden_size, not 4 * hidden_size",
    },
]

print("Common Pitfalls When Implementing vLLM Models")
print("=" * 60)
for i, p in enumerate(pitfalls, 1):
    print(f"\n{i}. {p['pitfall']}")
    print(f"   Symptom: {p['symptom']}")
    print(f"   Fix: {p['fix']}")

## 13. Debugging Workflow

In [ ]:
DEBUG_WORKFLOW = '''
# Step-by-step debugging workflow for a new model

# 1. Compare parameter names
import torch
from transformers import AutoModelForCausalLM

# Load HuggingFace model
hf_model = AutoModelForCausalLM.from_pretrained("path/to/model")
hf_params = set(hf_model.state_dict().keys())

# Load vLLM model
vllm_model = MyModelForCausalLM(config)
vllm_params = set(dict(vllm_model.named_parameters()).keys())

# Find mismatches
print("In HF but not vLLM:", hf_params - vllm_params)
print("In vLLM but not HF:", vllm_params - hf_params)

# 2. Test forward pass shape compatibility
input_ids = torch.randint(0, 1000, (10,))
positions = torch.arange(10)

with torch.no_grad():
    output = vllm_model(input_ids, positions, kv_caches=[], attn_metadata=None)
    print(f"Output shape: {output.shape}")
    assert output.shape[-1] == config.vocab_size

# 3. Compare outputs with HuggingFace
hf_output = hf_model(input_ids.unsqueeze(0)).logits.squeeze(0)
vllm_output = vllm_model(input_ids, positions, kv_caches=[], attn_metadata=None)

# Check numerical similarity
diff = (hf_output - vllm_output).abs().max()
print(f"Max absolute difference: {diff:.6f}")
assert diff < 1e-4, "Outputs differ too much!"

# 4. Test weight loading
from vllm.model_executor.model_loader.weight_utils import hf_model_weights_iterator

weights_iter = hf_model_weights_iterator("path/to/model")
loaded = vllm_model.load_weights(weights_iter)
print(f"Loaded {len(loaded)} weights")
'''

print(DEBUG_WORKFLOW)

## 14. The `prefix` Parameter for Pipeline Parallelism

When using Pipeline Parallelism (PP), different layers run on different GPUs. The `prefix` parameter helps with weight loading.

In [ ]:
# Pipeline Parallelism support

print("Pipeline Parallelism (PP) in vLLM")
print("=" * 60)
print("""
With PP=2 and 32 layers:
  GPU 0: embed_tokens + layers 0-15 
  GPU 1: layers 16-31 + norm + lm_head

Each GPU only loads its own layers' weights.

To support PP, your model needs:
1. make_empty_intermediate_tensors() method
2. Handle intermediate_tensors in forward()

class MyModelForCausalLM(nn.Module):
    
    def make_empty_intermediate_tensors(self, batch_size, dtype, device):
        return IntermediateTensors({
            "hidden_states": torch.zeros(
                (batch_size, self.config.hidden_size),
                dtype=dtype, device=device
            ),
            "residual": torch.zeros(
                (batch_size, self.config.hidden_size),
                dtype=dtype, device=device
            ),
        })
    
    def forward(self, input_ids, positions, kv_caches, 
                attn_metadata, intermediate_tensors=None):
        if intermediate_tensors is not None:
            # This GPU is NOT the first stage
            # Use intermediate tensors from previous GPU
            hidden_states = intermediate_tensors["hidden_states"]
        else:
            # This GPU IS the first stage
            hidden_states = self.model.embed_tokens(input_ids)
        
        # ... process layers ...
        
        if not is_last_stage:
            return IntermediateTensors({"hidden_states": hidden_states})
        else:
            return self.lm_head(self.model.norm(hidden_states))
""")

## 15. Exercises

### Exercise 1: Implement a GPT-2 Style Model
Implement a GPT-2 style model for vLLM with these differences from our Llama-style model:
- Uses `nn.LayerNorm` instead of `RMSNorm`
- Uses `nn.GELU` instead of `SiLU`
- Has bias in linear layers
- Uses absolute position embeddings instead of RoPE

### Exercise 2: Add GQA Support
Modify the attention to support Grouped Query Attention with an arbitrary `num_kv_heads`. Test with:
- `num_heads=32, num_kv_heads=32` (MHA)
- `num_heads=32, num_kv_heads=8` (GQA)
- `num_heads=32, num_kv_heads=1` (MQA)

### Exercise 3: Weight Loading Debug Challenge
Given a HuggingFace checkpoint with these weight names, write the mapping:
```
transformer.wte.weight
transformer.h.0.attn.c_attn.weight  (fused QKV)
transformer.h.0.attn.c_proj.weight
transformer.h.0.mlp.c_fc.weight
transformer.h.0.mlp.c_proj.weight
transformer.ln_f.weight
lm_head.weight
```

In [ ]:
# Exercise 1: Starter code for GPT-2 style model

class GPT2StyleMLP(nn.Module):
    """GPT-2 style MLP (no gating, uses GELU)."""
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        # TODO: Implement with bias=True and GELU activation
        # Hint: No gate_up merging needed - just fc_in and fc_out
        self.c_fc = nn.Linear(hidden_size, intermediate_size, bias=True)
        self.c_proj = nn.Linear(intermediate_size, hidden_size, bias=True)
        self.act = nn.GELU()
    
    def forward(self, x):
        # TODO: Implement forward pass
        return self.c_proj(self.act(self.c_fc(x)))

# Test
mlp = GPT2StyleMLP(256, 1024)
x = torch.randn(4, 8, 256)
out = mlp(x)
print(f"GPT-2 MLP: {x.shape} -> {out.shape}")

## Summary

In this notebook, we implemented a complete model for vLLM from scratch:

1. **MLP**: SwiGLU with merged gate_up_proj using MergedColumnParallelLinear
2. **Attention**: GQA with QKVParallelLinear, RoPE, and vLLM's Attention wrapper
3. **Decoder Layer**: Pre-norm transformer block with residual connections
4. **Full Model**: Embedding + layers + norm + lm_head
5. **Weight Loading**: Handling stacked params, name mapping, and TP sharding
6. **Registration**: Using ModelRegistry.register_model()

### Key Takeaways
- Use vLLM's parallel linear layers for TP support
- Use the Attention wrapper for KV cache management
- Handle weight merging (QKV, gate_up) in load_weights()
- Test against HuggingFace for numerical correctness

### Next: Chapter 8.3 -- Adding Quantization Support